# 4. Ensembles de Arboles de Decision

## 4.06 GBDT LightGBM

La técnica de Gradient Boosting fue creada por Jerome H. Friedman en 1999 - 2001
<br>Se implementaron librerías ineficientes
<br>En 2016 se crea XGBoost, en 2017 LightGBM




Paper original de  Gradient Boosting


Friedman JH. Greedy function approximation: A gradient
boosting machine. Ann Stat. 2001;29(5):1189–232. https://
doi.org/10.1214/aos/1013203451.
<br>
https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.pdf

Paper XGBoost

Chen, T.; Guestrin, C. Xgboost: A scalable tree boosting system. In Proceedings of the 22nd ACM Sigkdd International Conference on Knowledge Discovery and Data Mining, San Francisco, CA, USA, 13–17 August 2016; pp. 785–794.

https://dl.acm.org/doi/pdf/10.1145/2939672.2939785

Paper  LightGBM

Ke G., Meng Q., Finley T., Wang T., Chen W., Ma W., et al.
Lightgbm: A highly efficient gradient boosting decision tree
Advances in Neural Information Processing Systems, 30 (2017)

https://proceedings.neurips.cc/paper/2017/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf

Videos "simplificados" :
*  https://www.youtube.com/watch?v=3CC4N4z3GJc
*  https://www.youtube.com/watch?v=2xudPOBz-vs
*  https://www.youtube.com/watch?v=jxuNLH5dXCs
*  https://www.youtube.com/watch?v=StWY5QWMXCw

Artículos ligeros:
*  https://machinelearningmastery.com/gentle-introduction-gradient-boosting-algorithm-machine-learning/
*   https://www.machinelearningplus.com/machine-learning/an-introduction-to-gradient-boosting-decision-trees/
*   https://medium.com/@ruchi.awasthi63/gradient-boosted-decision-tree-clearly-explained-bd1d8c7d9923
*   https://medium.com/data-science/a-visual-understanding-of-decision-trees-and-gradient-boosting-c6bc53f982ce
*   https://medium.com/analytics-vidhya/introduction-to-the-gradient-boosting-algorithm-c25c653f826b
*   https://medium.com/@datasciencewizards/understanding-the-gradient-boosting-algorithm-9fe698a352ad

El Gradient Boosting of Decision Trees es un ensemble de árboles de decisión, para un nuevo registro la predicción se hace sumando el score que cada arbol asigna a ese registro.

En GBDT la construccion de los árboles es secuencial, ya que el arbol n-simo se genera para predecir el error del modelo conformado por los n-1 arboles previos, aunque sea un arbol de clasificación lo que se predice es un numero real mediante un arbol de regresión.

<br>Qué tipo de perturbaciones se realiza LightGBM

*   Se perturba el dataset, seleccionando para cada arbol un subconjunto de las columnas.
*   El algortimo de arbol de decisión no presenta perturbaciones

Cada arbolito de LightGBM se entrena sobre un dataset perturbado, que en principio posee :
* todos los registros del dataset original
* solo un porcentaje *feature_fraction* de las columnas originales del dataset

#### 4.06.1  Seteo del ambiente local


Esta parte se corre con el kernel de **R** (local).


Los archivos persistentes quedan en el repo local: datasets en `datasets/` y resultados en `exp/`.


In [1]:
# Seteo local: resuelve el root del repo (funciona desde raiz, src/ensembles o exp/<exp>)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")


Para correr localmente, el dataset debe estar en `datasets/` dentro del repo.

<br>Si se va a subir a Kaggle, copiar `kaggle.json` a la raiz del repo antes de correr la siguiente celda. La celda lo instala en `~/.kaggle/kaggle.json` con permisos correctos.


In [2]:
dir.create(DATA_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)

dataset_local <- file.path(DATA_DIR, "dataset_pequeno.csv")
if (!file.exists(dataset_local)) {
  stop("No encuentro el dataset en: ", dataset_local)
}

if (file.exists(KAGGLE_JSON)) {
  kaggle_dir <- path.expand("~/.kaggle")
  dir.create(kaggle_dir, recursive = TRUE, showWarnings = FALSE)
  file.copy(KAGGLE_JSON, file.path(kaggle_dir, "kaggle.json"), overwrite = TRUE)
  Sys.chmod(file.path(kaggle_dir, "kaggle.json"), mode = "0600")
} else {
  message("No encontre kaggle.json en la raiz del repo. Solo es necesario para hacer submit a Kaggle.")
}


### 4.07  LightGBM, una corrida

Esta parte se debe correr con el kernel de **R**.


limpio el ambiente de R

In [3]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 15 10:44:46 2026"

In [4]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,676829,36.2,1489787,79.6,NA,1489787,79.6
Vcells,1259246,9.7,8388608,64.0,49152,2014488,15.4


In [5]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: rlist

Loading required package: lightgbm

Warning message:
“package ‘lightgbm’ was built under R version 4.6.1”


Aqui debe cargar SU semilla primigenia

In [6]:
PARAM <- list()
PARAM$experimento <- 5957
PARAM$semilla_primigenia <- 300089

# estos hiperparametros de LightGBM surgieron de una Bayesian Optimization
# PARAM$lgb$num_iterations <- 1000  # cantidad de arbolitos
# PARAM$lgb$learning_rate <- 0.027
# PARAM$lgb$feature_fraction <- 0.8
# PARAM$lgb$min_data_in_leaf <- 76
# PARAM$lgb$num_leaves <- 8
# PARAM$lgb$max_bin <- 31


# BUEN COMBO ESTE
# PARAM$lgb$num_iterations <- 2200
# PARAM$lgb$learning_rate <- 0.005
# PARAM$lgb$feature_fraction <- 0.7
# PARAM$lgb$min_data_in_leaf <- 500
# PARAM$lgb$num_leaves <- 10
# PARAM$lgb$max_bin <- 500


PARAM$lgb$num_iterations <- 2200
PARAM$lgb$learning_rate <- 0.005
PARAM$lgb$feature_fraction <- 0.7
PARAM$lgb$min_data_in_leaf <- 500
PARAM$lgb$num_leaves <- 10
PARAM$lgb$max_bin <- 500


In [7]:
# carpeta de trabajo (re-resuelve paths por si hubo rm() arriba)
resolve_root <- function(start = getwd()) {
  d <- normalizePath(start, mustWork = TRUE)
  for (i in seq_len(8)) {
    if (dir.exists(file.path(d, "datasets"))) return(d)
    parent <- dirname(d)
    if (identical(parent, d)) break
    d <- parent
  }
  stop("No encuentro la carpeta datasets/ subiendo desde: ", start)
}

ROOT_DIR <- resolve_root()
DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")

dir.create(EXP_DIR, recursive = TRUE, showWarnings = FALSE)
experimento_folder <- paste0("KA", PARAM$experimento)
dir.create(file.path(EXP_DIR, experimento_folder), recursive = TRUE, showWarnings = FALSE)
setwd(file.path(EXP_DIR, experimento_folder))


In [8]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"), stringsAsFactors = TRUE)


In [9]:
# paso la clase a binaria

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+2"), 1L, 0L)]

In [10]:
# los campos que se van a utilizar

campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01"))

In [11]:
# establezco donde entreno

dataset[, train := 0L]
dataset[foto_mes %in% c(202107), train := 1L]

In [12]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[train == 1L, campos_buenos, with= FALSE]),
  label= dataset[train == 1L, clase01]
)

In [13]:
# genero el modelo
# estos hiperparametros  salieron de una laaarga Optmizacion Bayesiana

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG") # Establezco la semilla aleatoria

modelo <- lgb.train(
  data= dtrain,
  param= list(
    objective= "binary",
    max_bin= PARAM$lgb$max_bin,
    learning_rate= PARAM$lgb$learning_rate,
    num_iterations= PARAM$lgb$num_iterations,
    num_leaves= PARAM$lgb$num_leaves,
    min_data_in_leaf= PARAM$lgb$min_data_in_leaf,
    feature_fraction= PARAM$lgb$feature_fraction,
    seed= PARAM$semilla_primigenia
  )
)


[LightGBM] [Info] Number of positive: 1304, number of negative: 163175
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.098423 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34511
[LightGBM] [Info] Number of data points in the train set: 164479, number of used features: 142
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.007928 -> initscore=-4.829387
[LightGBM] [Info] Start training from score -4.829387


In [14]:
# ahora imprimo la importancia de variables
tb_importancia <- as.data.table(lgb.importance(modelo))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)


In [15]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo, "modelo.txt" )

In [16]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)


In [17]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

In [18]:
# subidas a Kaggle
# ordeno por probabilidad descendente

setorder(tb_prediccion, -prob)

In [19]:
# genero la prediccion y subo a Kaggle

tb_prediccion[, Predicted := 0L]
tb_prediccion[prob>(1/40), Predicted := 1L]

archivo_kaggle <- paste0("KA", PARAM$experimento, ".csv")

# grabo el archivo
fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
  file= archivo_kaggle,
  sep= ","
)

# subida a Kaggle
comando <- "kaggle competitions submit"
competencia <- "-c data-mining-inicial-2026-b"
arch <- paste( "-f", archivo_kaggle)

mensaje <- paste0("-m 'num_iterations=", PARAM$lgb$num_iterations,
  "  learning_rate=", PARAM$lgb$learning_rate,
  "  feature_fraction=", PARAM$lgb$feature_fraction,
  "  min_data_in_leaf=", PARAM$lgb$min_data_in_leaf,
  "  num_leaves=",PARAM$lgb$num_leaves,
  "  max_bin=", PARAM$lgb$max_bin,
"'" )

linea <- paste( comando, competencia, arch, mensaje)
salida <- system(linea, intern=TRUE)
cat(salida)

97 submissions remaining today. Successfully submitted to Data Mining, Inicial 2026 B

In [20]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 15 10:45:35 2026"



---

